In [1]:
import os, time, requests
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

cities = {
    "Amsterdam": {"lat": 52.3676, "lon": 4.9041,  "country": "NLD"},
    "Barcelona": {"lat": 41.3874, "lon": 2.1686,  "country": "ESP"},
    "Berlin":    {"lat": 52.5200, "lon": 13.4050, "country": "DEU"},
    "Columbus":  {"lat": 39.9612, "lon": -82.9988, "country": "USA"},
}

In [2]:
import math

HEADERS = {"User-Agent": "airbnb-student-project/1.0 (contact:rofaidaesmail234@gmail.com)"}
URL = "https://en.wikipedia.org/w/api.php"

def offset(lat, lon, dn_km, de_km):
    return (lat + dn_km / 111.0, lon + de_km / (111.0 * math.cos(math.radians(lat))))

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

STEPS = [(0, 0), (5, 0), (-5, 0), (0, 5), (0, -5), (5, 5), (5, -5), (-5, 5), (-5, -5)]

rows = []
for city, c in cities.items():
    for dn, de in STEPS:
        lat, lon = offset(c["lat"], c["lon"], dn, de)
        params = {"action": "query", "list": "geosearch", "format": "json",
                  "gscoord": f"{lat}|{lon}", "gsradius": 10000, "gslimit": 500}
        r = requests.get(URL, params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        for p in r.json()["query"]["geosearch"]:
            rows.append({"city": city, "name": p["title"],
                         "latitude": p["lat"], "longitude": p["lon"],
                         "source_url": f"https://en.wikipedia.org/?curid={p['pageid']}"})
        time.sleep(1)

landmarks = pd.DataFrame(rows).drop_duplicates(subset=["city", "source_url"])
landmarks["distance_from_center_m"] = [
    round(haversine_m(cities[ci]["lat"], cities[ci]["lon"], la, lo), 1)
    for ci, la, lo in zip(landmarks["city"], landmarks["latitude"], landmarks["longitude"])]
landmarks = landmarks[landmarks["distance_from_center_m"] <= 10000]
print(landmarks.groupby("city").size())

city
Amsterdam     679
Barcelona     735
Berlin       1158
Columbus      616
dtype: int64


In [3]:
landmarks.groupby("city").head(15)[["city", "name", "distance_from_center_m"]]

,city,name,distance_from_center_m
0,Amsterdam,Joods Historisch Museum,73.1
1,Amsterdam,Mozes en Aäronkerk,87.1
2,Amsterdam,Portuguese Synagogue (Amsterdam),89.0
3,Amsterdam,Netherlands Film Academy,91.8
4,Amsterdam,Leprozengracht,124.1
5,Amsterdam,Markenplein,137.8
6,Amsterdam,Nieuwe Herengracht,168.8
7,Amsterdam,Stopera,170.5
8,Amsterdam,Waterlooplein metro station,170.5
9,Amsterdam,Houtgracht,172.8


In [4]:
exclude = [
    "station", "metro", "line ", "street", "straße", "strasse", "carrer", "gracht",
    "avenue", "avinguda", "road", "crash", "bombing", "protest", "consulate",
    "embassy", "academy", "university", "school", "district", "locality",
    "(locality)", "east berlin", "west berlin", "list of", "history of",
]
pattern_ex = "|".join(exclude)

mask = ~landmarks["name"].str.lower().str.contains(pattern_ex, na=False)
mask &= landmarks["name"].str.lower() != landmarks["city"].str.lower()   # شيلي صفحة المدينة نفسها
clean_lm = landmarks[mask].copy()
print(clean_lm.groupby("city").size())
clean_lm.groupby("city").head(15)[["city", "name", "distance_from_center_m"]]

city
Amsterdam    566
Barcelona    517
Berlin       924
Columbus     493
dtype: int64


C:\Users\rokas\AppData\Local\Temp\ipykernel_28372\3943134667.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = ~landmarks["name"].str.lower().str.contains(pattern_ex, na=False)


,city,name,distance_from_center_m
0,Amsterdam,Joods Historisch Museum,73.1
1,Amsterdam,Mozes en Aäronkerk,87.1
2,Amsterdam,Portuguese Synagogue (Amsterdam),89.0
5,Amsterdam,Markenplein,137.8
7,Amsterdam,Stopera,170.5
10,Amsterdam,290 Square Meters,181.1
11,Amsterdam,Waterlooplein,184.2
12,Amsterdam,Reinwardt Academie,185.9
13,Amsterdam,Jodenbreestraat,188.1
14,Amsterdam,Maupoleum,202.8


In [5]:
clean_lm.to_sql("raw_landmarks", engine, if_exists="replace", index=False)
with engine.connect() as conn:
    print(conn.execute(text("SELECT city, COUNT(*) FROM raw_landmarks GROUP BY city")).fetchall())

[('Amsterdam', 566), ('Barcelona', 517), ('Berlin', 924), ('Columbus', 493)]


In [6]:
clean_lm.to_sql("raw_landmarks", engine, if_exists="replace", index=False)
with engine.connect() as conn:
    print(conn.execute(text("SELECT city, COUNT(*) FROM raw_landmarks GROUP BY city")).fetchall())

[('Amsterdam', 566), ('Barcelona', 517), ('Berlin', 924), ('Columbus', 493)]
